# DatasetMain Integrity Check

Use this notebook to validate every dataset under `/playpen-ssd/smerrill/deception2/DatasetMain`.

It checks:
- `examples.jsonl` line-by-line JSON validity
- `sentences.jsonl` line-by-line JSON validity
- `manifest.json` readability
- localization output counts for quick context

Broken or incomplete datasets are shown first, followed by a full summary table.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 160)

REPO_ROOT = Path("/playpen-ssd/smerrill/deception2")
DATASET_ROOT = REPO_ROOT / "DatasetMain"
DATASET_ROOT


In [ ]:
def safe_relpath(path: Path, root: Path = REPO_ROOT) -> str:
    try:
        return str(path.relative_to(root))
    except ValueError:
        return str(path)


def validate_jsonl(path: Path, preview_chars: int = 240) -> dict[str, Any]:
    info: dict[str, Any] = {
        "path": safe_relpath(path),
        "exists": path.exists(),
        "status": "missing",
        "valid_lines": 0,
        "bad_line": None,
        "error": None,
        "preview": None,
        "size_mb": round(path.stat().st_size / (1024 * 1024), 2) if path.exists() else None,
        "mtime": pd.Timestamp(path.stat().st_mtime, unit="s") if path.exists() else None,
    }
    if not path.exists():
        return info

    try:
        with path.open("r", encoding="utf-8", errors="replace") as f:
            for line_no, raw_line in enumerate(f, 1):
                if not raw_line.strip():
                    continue
                try:
                    json.loads(raw_line)
                except Exception as exc:
                    info["status"] = "bad_json"
                    info["bad_line"] = line_no
                    info["error"] = f"{type(exc).__name__}: {exc}"
                    info["preview"] = raw_line[:preview_chars].replace("\n", "\\n")
                    return info
                info["valid_lines"] += 1
    except Exception as exc:
        info["status"] = "read_error"
        info["error"] = f"{type(exc).__name__}: {exc}"
        return info

    info["status"] = "ok"
    return info


def validate_manifest(path: Path) -> dict[str, Any]:
    info: dict[str, Any] = {
        "path": safe_relpath(path),
        "exists": path.exists(),
        "status": "missing",
        "error": None,
    }
    if not path.exists():
        return info

    try:
        json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        info["status"] = "bad_json"
        info["error"] = f"{type(exc).__name__}: {exc}"
        return info

    info["status"] = "ok"
    return info


def count_localization_outputs(dataset_dir: Path) -> int:
    loc_dir = dataset_dir / "localization"
    if not loc_dir.exists():
        return 0
    return sum(1 for _ in loc_dir.glob("sentence_localization_*.json"))


def iter_dataset_dirs(dataset_root: Path) -> list[Path]:
    dataset_dirs: list[Path] = []
    if not dataset_root.exists():
        return dataset_dirs

    for env_dir in sorted(dataset_root.iterdir()):
        if not env_dir.is_dir():
            continue
        for model_dir in sorted(env_dir.iterdir()):
            if model_dir.is_dir():
                dataset_dirs.append(model_dir)
    return dataset_dirs


def validate_dataset_dir(dataset_dir: Path) -> dict[str, Any]:
    examples = validate_jsonl(dataset_dir / "examples.jsonl")
    sentences = validate_jsonl(dataset_dir / "sentences.jsonl")
    manifest = validate_manifest(dataset_dir / "manifest.json")

    issues: list[str] = []
    for label, info in (("examples", examples), ("sentences", sentences), ("manifest", manifest)):
        if info["status"] == "ok":
            continue
        issue = f"{label}:{info['status']}"
        if info.get("bad_line") is not None:
            issue += f"@{info['bad_line']}"
        issues.append(issue)

    return {
        "dataset": str(dataset_dir.relative_to(DATASET_ROOT)),
        "env": dataset_dir.parent.name,
        "model_tail": dataset_dir.name,
        "overall_status": "ok" if not issues else "issue",
        "examples_status": examples["status"],
        "examples_lines": examples["valid_lines"],
        "examples_bad_line": examples["bad_line"],
        "sentences_status": sentences["status"],
        "sentences_lines": sentences["valid_lines"],
        "sentences_bad_line": sentences["bad_line"],
        "manifest_status": manifest["status"],
        "localization_files": count_localization_outputs(dataset_dir),
        "notes": "; ".join(issues) if issues else "ok",
        "examples_path": examples["path"],
        "sentences_path": sentences["path"],
    }


In [ ]:
rows = [validate_dataset_dir(dataset_dir) for dataset_dir in iter_dataset_dirs(DATASET_ROOT)]
summary_df = pd.DataFrame(rows)

if summary_df.empty:
    display(Markdown(f"No datasets found under `{DATASET_ROOT}`."))
else:
    sort_rank = {"issue": 0, "ok": 1}
    summary_df["_sort_rank"] = summary_df["overall_status"].map(sort_rank).fillna(99)
    summary_df = summary_df.sort_values(["_sort_rank", "env", "model_tail"]).drop(columns=["_sort_rank"])
    broken_df = summary_df[summary_df["overall_status"] != "ok"].copy()

    display(Markdown(f"## Broken Or Incomplete Datasets ({len(broken_df)})"))
    if broken_df.empty:
        display(Markdown("All datasets passed validation."))
    else:
        display(broken_df)

    display(Markdown("## Status Counts By Environment"))
    display(summary_df.groupby(["env", "overall_status"]).size().unstack(fill_value=0))

    display(Markdown(f"## Full Dataset Summary ({len(summary_df)})"))
    display(summary_df)


In [ ]:
TARGET_DATASET = broken_df.iloc[0]["dataset"] if "broken_df" in globals() and not broken_df.empty else None
TARGET_DATASET


In [ ]:
def inspect_dataset(dataset_key: str | None) -> None:
    if not dataset_key:
        display(Markdown("No broken dataset selected for detailed inspection."))
        return

    dataset_dir = DATASET_ROOT / dataset_key
    if not dataset_dir.exists():
        display(Markdown(f"Dataset not found: `{dataset_key}`"))
        return

    checks = {
        "examples.jsonl": validate_jsonl(dataset_dir / "examples.jsonl", preview_chars=400),
        "sentences.jsonl": validate_jsonl(dataset_dir / "sentences.jsonl", preview_chars=400),
        "manifest.json": validate_manifest(dataset_dir / "manifest.json"),
    }

    records = []
    for file_name, info in checks.items():
        records.append(
            {
                "file": file_name,
                "status": info.get("status"),
                "bad_line": info.get("bad_line"),
                "error": info.get("error"),
                "path": info.get("path"),
            }
        )

    display(Markdown(f"## Detailed Inspection: `{dataset_key}`"))
    display(pd.DataFrame(records))

    for file_name, info in checks.items():
        preview = info.get("preview")
        if preview:
            display(Markdown(f"### {file_name} first bad line preview"))
            print(preview)


inspect_dataset(TARGET_DATASET)
